# Least squares · Module 03
Python 3 and NumPy only. Write A and b on paper before computing. Use `np.linalg.lstsq`; do not invert `A.T @ A`.


In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(203)
t = np.array([0., 1., 2.]); b = np.array([1., 2., 2.])
A = np.column_stack([t, np.ones_like(t)])
theta, _, rank, singular = np.linalg.lstsq(A, b, rcond=None)
residual = A @ theta - b
print('slope, intercept:', theta, 'rank:', rank, 'singular values:', singular)
assert np.allclose(theta, [.5, 7 / 6])
print('A.T @ residual (should be zero):', A.T @ residual)
assert np.allclose(A.T @ residual, 0, atol=1e-12)


slope, intercept: [0.5    1.1667] rank: 2 singular values: [2.6762 0.9153]
A.T @ residual (should be zero): [-0. -0.]


## Part 2: Clean, noisy and one outlier
Fit the same line to three versions of the data. Squared error punishes a large residual heavily, so one bad point can pull the whole fit. Predict which coefficient moves most before you run the cell.


In [2]:
t = np.linspace(0, 10, 25); truth = np.array([2., -1.])
A = np.column_stack([t, np.ones_like(t)])
clean = A @ truth
noisy = clean + rng.normal(0, .5, t.size)
outlier = noisy.copy(); outlier[-1] += 30
results = {}
for name, b in [('clean', clean), ('noisy', noisy), ('one outlier', outlier)]:
    fit = np.linalg.lstsq(A, b, rcond=None)[0]
    r = A @ fit - b
    results[name] = fit
    print(f'{name:12s} fit {fit}  RMSE {np.sqrt(np.mean(r ** 2)):.3f}  largest |r| {np.abs(r).max():.2f}')
assert np.allclose(results['clean'], truth)
assert np.linalg.norm(results['one outlier'] - truth) > 3 * np.linalg.norm(results['noisy'] - truth)


clean        fit [ 2. -1.]  RMSE 0.000  largest |r| 0.00
noisy        fit [ 1.9922 -0.9245]  RMSE 0.617  largest |r| 1.07
one outlier  fit [ 2.6568 -3.0475]  RMSE 5.531  largest |r| 25.33


## Part 3: Why stable solvers matter
Forming `A.T @ A` squares the condition number. QR works on A directly. Here a polynomial design matrix is badly conditioned, so the two routes give visibly different accuracy on a problem whose answer is known.


In [3]:
t = np.linspace(0, 1, 40); A = np.vander(t, 10, increasing=True)
known = rng.normal(0, 1, 10); b = A @ known
print('condition of A:', np.linalg.cond(A), ' of A.T@A:', np.linalg.cond(A.T @ A))
normal = np.linalg.solve(A.T @ A, A.T @ b)
Q, R = np.linalg.qr(A); qr = np.linalg.solve(R, Q.T @ b)
errors = np.linalg.norm(normal - known), np.linalg.norm(qr - known)
print('coefficient error, normal equations:', errors[0], ' QR:', errors[1])
assert errors[1] < errors[0]
print('explicit inverse for a small singular matrix fails:', end=' ')
try: np.linalg.inv(np.array([[1., 1.], [2., 2.]]))
except np.linalg.LinAlgError as error: print(error)
print('pseudoinverse still gives a minimum-norm answer:', np.linalg.pinv(np.array([[1., 1.], [2., 2.]])) @ np.array([1., 2.]))


condition of A: 3520666.470089475  of A.T@A: 12395183693051.123
coefficient error, normal equations: 0.0006878937424447963  QR: 5.531035966949341e-11
explicit inverse for a small singular matrix fails: Singular matrix
pseudoinverse still gives a minimum-norm answer: [0.5 0.5]


## Part 4: Ridge regression and an honest choice of lambda
A flexible polynomial can chase noise. A penalty pulls the coefficients down. Choose lambda using validation error, then touch the test set once. The intercept is left unpenalized.


In [4]:
def design(x): return np.vander(x, 10, increasing=True)
def sample(n):
    x = rng.uniform(-1, 1, n)
    return x, np.sin(3 * x) + rng.normal(0, .25, n)
xtr, ytr = sample(16); xval, yval = sample(60); xtest, ytest = sample(200)
Atr, Aval, Atest = design(xtr), design(xval), design(xtest)
def ridge(A, y, lam):
    penalty = np.sqrt(lam) * np.eye(A.shape[1])[1:]
    return np.linalg.lstsq(np.vstack([A, penalty]), np.r_[y, np.zeros(A.shape[1] - 1)], rcond=None)[0]
mse = lambda A, y, w: np.mean((A @ w - y) ** 2)
choices = [0., 1e-4, 1e-2, 1e-1, 1., 10.]; scores = []
for lam in choices:
    w = ridge(Atr, ytr, lam); scores.append(mse(Aval, yval, w))
    print(f'lambda {lam:<7} train MSE {mse(Atr, ytr, w):.4f}  validation MSE {scores[-1]:.4f}  weight norm {np.linalg.norm(w):8.2f}')
best = choices[int(np.argmin(scores))]
w = ridge(Atr, ytr, best)
print('selected lambda:', best, ' test MSE, used once:', mse(Atest, ytest, w), ' test count:', ytest.size)
assert best > 0 and min(scores) < scores[0]


lambda 0.0     train MSE 0.0124  validation MSE 2.7498  weight norm    85.48
lambda 0.0001  train MSE 0.0134  validation MSE 0.0635  weight norm     6.60
lambda 0.01    train MSE 0.0163  validation MSE 0.0445  weight norm     3.64
lambda 0.1     train MSE 0.0307  validation MSE 0.0597  weight norm     2.57
lambda 1.0     train MSE 0.1463  validation MSE 0.1662  weight norm     1.24
lambda 10.0    train MSE 0.3699  validation MSE 0.4335  weight norm     0.38
selected lambda: 0.01  test MSE, used once: 0.06131734206020305  test count: 200


## Your experiment
Add a second outlier, or move the outlier to the middle of the range, and explain why the effect changes. Then repeat Part 4 with a fresh seed and report whether the chosen lambda is stable. Your fitting report should compare clean, noisy and outlier fits, separate residual from measurement noise, and state what the fit cannot tell you.
